In [25]:
import pandas as pd

df = pd.read_csv("/workspaces/project.final_allocation/notebook/ambulance_combined_dataset 2.csv")
df

,datetime,latitude,longitude,emergency_type,hour,day,traffic_level,temperature,rain,demand
0,10-12-2015 17:40,40.297876,-75.581294,EMS,17,Thursday,0.562178,38,0,1
1,10-12-2015 17:40,40.258061,-75.264680,EMS,17,Thursday,0.965500,21,0,1
2,10-12-2015 17:40,40.121182,-75.351975,Fire,17,Thursday,0.812396,20,0,1
3,10-12-2015 17:40,40.116153,-75.343513,EMS,17,Thursday,0.719061,33,0,1
4,10-12-2015 17:40,40.251492,-75.603350,EMS,17,Thursday,0.409213,21,0,1
...,...,...,...,...,...,...,...,...,...,...
99487,24-08-2016 11:06,40.132869,-75.333515,Traffic,11,Wednesday,0.833602,25,1,1
99488,24-08-2016 11:07,40.006974,-75.289080,Traffic,11,Wednesday,0.359906,29,0,1
99489,24-08-2016 11:12,40.115429,-75.334679,EMS,11,Wednesday,0.601142,25,0,1
99490,24-08-2016 11:17,40.186431,-75.192555,EMS,11,Wednesday,0.594402,38,0,1


In [26]:
df.isna().sum()

datetime          0
latitude          0
longitude         0
emergency_type    0
hour              0
day               0
traffic_level     0
temperature       0
rain              0
demand            0
dtype: int64

In [27]:
df.to_csv("/workspaces/project.final_allocation/notebook/ambulance_updated_dataset.csv", index=False)
df

,datetime,latitude,longitude,emergency_type,hour,day,traffic_level,temperature,rain,demand
0,10-12-2015 17:40,40.297876,-75.581294,EMS,17,Thursday,0.562178,38,0,1
1,10-12-2015 17:40,40.258061,-75.264680,EMS,17,Thursday,0.965500,21,0,1
2,10-12-2015 17:40,40.121182,-75.351975,Fire,17,Thursday,0.812396,20,0,1
3,10-12-2015 17:40,40.116153,-75.343513,EMS,17,Thursday,0.719061,33,0,1
4,10-12-2015 17:40,40.251492,-75.603350,EMS,17,Thursday,0.409213,21,0,1
...,...,...,...,...,...,...,...,...,...,...
99487,24-08-2016 11:06,40.132869,-75.333515,Traffic,11,Wednesday,0.833602,25,1,1
99488,24-08-2016 11:07,40.006974,-75.289080,Traffic,11,Wednesday,0.359906,29,0,1
99489,24-08-2016 11:12,40.115429,-75.334679,EMS,11,Wednesday,0.601142,25,0,1
99490,24-08-2016 11:17,40.186431,-75.192555,EMS,11,Wednesday,0.594402,38,0,1


In [28]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['emergency_type'] = le.fit_transform(df['emergency_type'])

In [29]:
df['datetime']=le.fit_transform(df['datetime'])
df['emergency_type']=le.fit_transform(df['emergency_type'])
df['day']=le.fit_transform(df['day'])

In [30]:
import numpy as np

In [31]:
df['priority'] = np.select(
    [
        df['emergency_type'] <= 2,   # LOW
        df['emergency_type'] <= 5,   # MEDIUM
        df['emergency_type'] > 5     # HIGH
    ],
    [0, 1, 2]
)

In [32]:
print(df['priority'].value_counts())

priority
0    99492
Name: count, dtype: int64


In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99492 entries, 0 to 99491
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   datetime        99492 non-null  int64  
 1   latitude        99492 non-null  float64
 2   longitude       99492 non-null  float64
 3   emergency_type  99492 non-null  int64  
 4   hour            99492 non-null  int64  
 5   day             99492 non-null  int64  
 6   traffic_level   99492 non-null  float64
 7   temperature     99492 non-null  int64  
 8   rain            99492 non-null  int64  
 9   demand          99492 non-null  int64  
 10  priority        99492 non-null  int64  
dtypes: float64(3), int64(8)
memory usage: 8.3 MB


In [34]:
df.columns

Index(['datetime', 'latitude', 'longitude', 'emergency_type', 'hour', 'day',
       'traffic_level', 'temperature', 'rain', 'demand', 'priority'],
      dtype='str')

In [35]:
X = df.drop(labels=['demand'], axis=1)
Y = df['demand']

In [36]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X['emergency_type'] = le.fit_transform(X['emergency_type'])

In [37]:
X['datetime']=le.fit_transform(X['datetime'])
X['emergency_type']=le.fit_transform(X['emergency_type'])
X['day']=le.fit_transform(X['day'])

In [38]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_pre = scaler.fit_transform(X)

In [39]:
from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(
    X_pre, Y, test_size=0.2, random_state=42
)

In [40]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(xtrain, ytrain)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [41]:
train_score = model.score(xtrain, ytrain)
test_score = model.score(xtest, ytest)

print("Train Accuracy:", train_score)
print("Test Accuracy:", test_score)

Train Accuracy: 0.9999497443242497
Test Accuracy: 0.9488919041157847


In [42]:
ytrain_pred = model.predict(xtrain)
ytest_pred = model.predict(xtest)

In [43]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(ytest, ytest_pred)
print(cm)

[[18760    74     1     0]
 [  861   119     0     0]
 [   77     1     2     0]
 [    3     0     0     1]]


In [44]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [45]:
import pickle
import numpy as np

def smart_prediction(data):
    
    # load model & scaler
    with open('/workspaces/project.final_allocation/notebook/model.pkl', 'rb') as f:
        model = pickle.load(f)
        
    with open('/workspaces/project.final_allocation/notebook/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)

    # scale input
    data_scaled = scaler.transform([data])

    # prediction
    demand = model.predict(data_scaled)[0]

    # 🚑 ambulance allocation logic
    if demand == 1 and data[3] > 0.7:
        ambulances = 3
        level = "HIGH"
    elif demand == 1:
        ambulances = 2
        level = "MEDIUM"
    else:
        ambulances = 1
        level = "LOW"

    # ⏱️ response time (simple logic)
    response_time = 10 + (data[3] * 10)   # traffic based

    return {
        "Demand": int(demand),
        "Priority": level,
        "Ambulances": ambulances,
        "Response Time": round(response_time, 2)
    }

In [46]:
latitude = 40.2
longitude = -75.3
hour = 18
traffic = 0.8
temperature = 32
rain = 5
demand = 1
priority = 3
day = 2
month = 5

sample = [latitude, longitude, hour, traffic, temperature , demand, priority , day , month,hour]

result=smart_prediction(sample)

print(result)

{'Demand': 1, 'Priority': 'HIGH', 'Ambulances': 3, 'Response Time': 18.0}


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
